# 07 Resilience: Timeout, Retry, and Fallback (OpenClaw, 2026)

## What This Lesson Is
Engineer resilient invocation flows that bound latency and recover from transient failures safely.

## Scientific Lens
- Concept: Bounded retries with fallback escalation policy.
- Measure: Success-within-budget rate under injected transient failures.
- Validity Limit: Aggressive retries can amplify load and degrade shared systems.


## How It Works
1. Model transient failure handling with deterministic retry/fallback logic.
2. Enforce max-attempt and latency budget constraints.
3. Run live invocation with short then longer timeout to observe behavior.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: runs real OpenClaw CLI operations when available; otherwise prints explicit skip guidance.


In [ ]:
# Deterministic Demo
attempt_budget = 3
outcomes = ["timeout", "timeout", "ok"]
history = []
for attempt in range(1, attempt_budget + 1):
    result = outcomes[attempt - 1]
    history.append((attempt, result))
    if result == "ok":
        break
print(history)
assert history[-1][1] == "ok"
assert len(history) <= attempt_budget


In [ ]:
# Live Demo
import shutil
import subprocess

if shutil.which("openclaw") is None:
    print("Skipping live resilience demo: openclaw CLI is not installed.")
else:
    prompt = "In one sentence, explain why retry backoff matters in production systems."
    short = ["openclaw", "agent", "--local", "--to", "+15555550123", "--message", prompt, "--timeout", "5"]
    long = ["openclaw", "agent", "--local", "--to", "+15555550123", "--message", prompt, "--timeout", "180"]
    for label, cmd in (("short-timeout", short), ("fallback-timeout", long)):
        print(f"[{label}] $", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True)
        print((proc.stdout or proc.stderr).strip()[:1200])


## Applied Labs
1. Add jitter to retry delays and compare with fixed delay strategy.
2. Track cumulative timeout budget and abort when threshold exceeded.
3. Implement provider fallback sequence (OpenAI -> Ollama) in deterministic simulation.

## Validation Checklist
- Retry loop has explicit max-attempt and explicit success condition.
- Fallback path triggers only after deterministic failure criteria.
- Live run shows difference between short and long timeout behaviors.

## Further Reading
- AWS backoff+jitter guidance: https://aws.amazon.com/builders-library/timeouts-retries-and-backoff-with-jitter/
- Google SRE cascading failures: https://sre.google/sre-book/addressing-cascading-failures/
- OpenClaw docs: https://docs.openclaw.ai
